# Lesson07. 用激光眼睛看世界 - LiDAR空间感知

**教学主题：** 学习使用LiDAR（激光雷达）进行空间感知。

**核心目标：** 理解LiDAR工作原理及避障功能。

**课程安排：**

- **前30分钟（揭秘）：** 动画展示LiDAR原理，讲解点云数据的含义。

- **后90分钟（实践出真知）：**
  - **读取数据：** 编写程序，读取LiDAR返回的距离数据。
  - **【核心挑战】智能制动：** 行走中遇到障碍物时自动制动停止。

## 7.1 导入依赖包和消息接口

In [ ]:
import time
import sys

# 导入宇树SDK通信模块
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
# 导入点云消息类型（LiDAR的原始点云数据）
from unitree_sdk2py.idl.sensor_msgs.msg.dds_ import PointCloud2_

## 7.2 获取并统计激光点云数据

点云（Point Cloud）是LiDAR采样的所有距离点的集合，每个点代表空间中的一个位置。

In [ ]:
# LiDAR点云数据发布的DDS话题
TOPIC_CLOUD = "rt/utlidar/cloud"

def point_cloud_handler(msg: PointCloud2_):
    """
    点云数据处理的回调函数。
    
    当新的点云数据到达时，此函数会被自动调用。
    """
    # 解析消息头部的时间戳信息
    sec = msg.header.stamp.sec        # 秒
    nanosec = msg.header.stamp.nanosec  # 纳秒
    
    # 计算点数量（width * height，对于无序点云height通常=1）
    point_count = msg.width * msg.height
    
    # 打印点云基本信息
    print("接收到一帧原始点云数据！")
    print(f"\t时间戳 = {sec}.{nanosec}")
    print(f"\t参考坐标系 = {msg.header.frame_id}")
    print(f"\t点数量 = {point_count}\n")


if __name__ == "__main__":
    import platform  # 平台模块，用于检测操作系统类型
    # 检测当前操作系统
    if platform.system() == 'Linux':
        # Ubuntu Linux 系统
        ChannelFactoryInitialize(0, "ens37")
    elif platform.system() == 'Windows':
        # Windows 系统
        ChannelFactoryInitialize(0)
    else:
        # 处理其他操作系统，例如 macOS
        print(f"当前系统 {platform.system()} 暂不支持。")

    # 创建点云话题订阅者并绑定回调函数
    subscriber = ChannelSubscriber(TOPIC_CLOUD, PointCloud2_)
    # Init参数：回调函数、消息队列长度=10
    subscriber.Init(point_cloud_handler, queueLen=10)

    # 保持程序运行以持续接收数据
    try:
        while True:
            time.sleep(10)
    except KeyboardInterrupt:
        print("程序已停止")
        subscriber.Close()

## 7.3 解析激光距离数据 - 障碍检测

In [ ]:
import time
import sys

# 导入宇树SDK通信模块
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
# 导入点云消息类型
from unitree_sdk2py.idl.geometry_msgs.msg.dds_ import PointStamped_

# LiDAR距离信息发布的DDS话题
TOPIC_RANGE_INFO = "rt/utlidar/range_info"

def range_info_handler(msg: PointStamped_):
    """
    处理激光雷达的距离信息回调函数。
    
    LiDAR检测周围的距离，提供前、左、右三个方向的距离数据。
    """
    # 提取时间戳信息
    sec = msg.header.stamp.sec
    nanosec = msg.header.stamp.nanosec
    frame_id = msg.header.frame_id
    
    # 提取距离信息（Point的x/y/z分别对应前/左/右方向）
    front_range = msg.point.x  # 前方距离（米）
    left_range = msg.point.y   # 左方距离（米）
    right_range = msg.point.z  # 右方距离（米）
    
    # 打印格式化的距离信息
    print("接收到激光距离数据！")
    print(f"\t时间戳 = {sec}.{nanosec}")
    print(f"\t参考坐标系 = {frame_id}")
    print(f"\t前方距离 = {front_range:.2f}米")
    print(f"\t左方距离 = {left_range:.2f}米")
    print(f"\t右方距离 = {right_range:.2f}米\n")
    
    # 【障碍检测】如果前方距离小于0.5米，发出警告
    if front_range < 0.5:
        print("⚠️ 警告：前方障碍物距离过近！建议停止或转向")


if __name__ == "__main__":
    import platform  # 平台模块，用于检测操作系统类型
    # 检测当前操作系统
    if platform.system() == 'Linux':
        # Ubuntu Linux 系统
        ChannelFactoryInitialize(0, "ens37")
    elif platform.system() == 'Windows':
        # Windows 系统
        ChannelFactoryInitialize(0)
    else:
        # 处理其他操作系统，例如 macOS
        print(f"当前系统 {platform.system()} 暂不支持。")
    
    # 创建距离话题订阅者并初始化
    subscriber = ChannelSubscriber(TOPIC_RANGE_INFO, PointStamped_)
    subscriber.Init(range_info_handler, 10)  # 队列长度=10
    
    try:
        # 保持程序运行以持续接收消息
        while True:
            time.sleep(10)
    except KeyboardInterrupt:
        print("程序被用户中断")
    finally:
        subscriber.Close()